# LEGO Minifigure Classification — Models Summary

Complete reference for all trained models, results, and evaluation.

## Models Overview

| Model | Classes | Backbone | Key Features | Best Val Acc | Test Acc | Status |
|-------|---------|----------|--------------|--------------|----------|--------|
| **Baseline** | 122 | ResNet50 | Standard training | - | - | Reference |
| **Option B** | 28 | EfficientNet-B2 | Category merge, balanced sampling | - | - | Stable |
| **B Improved** | 28 | EfficientNet-B2 | + Augmentation, Focal Loss, larger images | - | - | Stable |
| **V2** | 37 | EfficientNet-B4 | + Town split, BatchNorm improvements, TTA | - | - | Stable |
| **V3** | 28 | EfficientNet-B4 | + Hyperparameter tuning, 3 loss variants (Focal/ArcFace/SupCon) | - | - | Tuned |
| **V4 (Latest)** | 28 | ConvNeXt-Small | + Weighted ensemble, SWA, Optional ConvNeXt backbone | - | - | Optimized |
| **ResNet50 Top-20** | 20 | ResNet50 | Baseline on top-20 classes only | - | - | Reference |


## Setup & Imports

In [ ]:
import os
import json
import glob
import re
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display, HTML
import subprocess

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

BASE_DIR = '/home/test/big_data_assignment_2'

# Color scheme for models
COLORS = {
    'Baseline': '#e74c3c',
    'Option B': '#f39c12',
    'B Improved': '#2ecc71',
    'V2': '#3498db',
    'V3-Focal': '#9b59b6',
    'V3-ArcFace': '#8e44ad',
    'V3-SupCon': '#2ecc71',
    'V4': '#e67e22'
}

print("Setup complete. Navigate to desired analysis below.")

## Model Details

### V4 Optimized (Latest)
**File:** `optionB_v4_optimized.py`

Three targeted optimizations over V3:

1. **Weighted Model Ensemble**
   - Combines Focal, ArcFace, and SupCon model outputs
   - Weights by each model's validation F1
   - Typical improvement: +1-2% F1

2. **Stochastic Weight Averaging (SWA)**
   - Averages model weights after 75% of epochs
   - Finds flatter minima in loss landscape
   - Typical improvement: +1-3% F1 with no extra data/parameters

3. **Optional ConvNeXt-Small Backbone**
   - Modern alternative to EfficientNet-B4
   - Better ImageNet performance at similar cost
   - Toggle `USE_CONVNEXT = True` in config

**Config:**
- Classes: 28 (merged categories)
- Image size: 384 (ConvNeXt) / 380 (EfficientNet)
- Batch size: 16
- Epochs: 20
- Learning rate: 3e-4
- Label smoothing: 0.1
- Focal gamma: 2.0
- MixUp + CutMix augmentation

### V3 Hyperparameter Tuning
**File:** `optionB_v3_hpopt.py`

Tests 3 loss variants with Optuna hyperparameter optimization:
- Focal Loss
- ArcFace Loss
- Supervised Contrastive Loss

3-fold cross-validation on best hyperparameters.

### V2 (Baseline with Improvements)
**File:** `optionB_train.py` (original) or use latest code

Improvements over B Improved:
- EfficientNet-B4 backbone
- Town-based category splitting
- Enhanced batch normalization
- Test-time augmentation (TTA)

### Earlier Versions
- **Baseline:** Standard ResNet50, 122 classes
- **Option B:** Initial category merging, 28 classes
- **B Improved:** Augmentation + Focal Loss + larger images

See `optionB_v2.py`, `optionB_improved.py`, `optionB_train.py`, `baseline_train.py` for implementation details.

## Training & Evaluation

### Run Latest Model (V4)
```bash
python optionB_v4_optimized.py
```

### Run Hyperparameter Tuning (V3)
```bash
python optionB_v3_hpopt.py
```

### Run ResNet50 Baseline (Top-20 Classes)
```bash
python resnet50_simple.py
```

### Evaluate All Models
```bash
python evaluate_all.py
```

### View Results with Dashboard
Use `training_dashboard.ipynb` to monitor live training and visualize results.

## Results Visualization

In [ ]:
# Load results from result files if available
result_dirs = glob.glob(os.path.join(BASE_DIR, '*_results'))

html = '<h3>Available Result Directories:</h3><ul>'
for d in sorted(result_dirs):
    dirname = os.path.basename(d)
    files = os.listdir(d)
    html += f'<li><strong>{dirname}</strong> ({len(files)} files)</li>'
html += '</ul>'

display(HTML(html))

In [ ]:
# Summary statistics
print("Model Training Summary:")
print("=" * 60)
print()
print("Latest: V4 Optimized")
print("  - Weighted ensemble of 3 loss variants")
print("  - Stochastic weight averaging (SWA)")
print("  - Optional ConvNeXt-Small backbone")
print()
print("Previous versions (V3, V2, B Improved, Option B) available")
print("for reference and comparison.")
print()
print("See training logs and result directories for detailed metrics.")

## File Structure

```
big_data_assignment_2/
├── Models_Summary.ipynb              ← You are here (consolidated reference)
├── training_dashboard.ipynb          ← Live training monitor
├── optionB_v4_optimized.py           ← Latest model (run this)
├── optionB_v3_hpopt.py               ← Hyperparameter tuning
├── resnet50_simple.py                ← ResNet50 baseline
├── evaluate_all.py                   ← Batch evaluation
├── baseline_train.py, optionB_*.py   ← Previous versions (reference only)
├── optionB_v4_results/               ← V4 outputs
├── optionB_v3_hpopt_results/         ← V3 HP tuning results
├── resnet50_top20_results/           ← ResNet50 outputs
├── README.md                         ← Project overview
├── QUICKSTART.md                     ← Getting started
└── [other documentation]
```

## Quick Reference

**What to run for best results:**
```bash
python optionB_v4_optimized.py
```

**Monitor training:**
- Open `training_dashboard.ipynb` in Jupyter
- Re-run cells to refresh with latest metrics

**Compare all models:**
```bash
python evaluate_all.py
```

**Run hyperparameter search:**
```bash
python optionB_v3_hpopt.py  # Long-running, tests 30 HP combinations
```

For full documentation, see README.md and QUICKSTART.md.